# LoD1 Timing and Quality Considerations

[![Binder](_static/launch-binder.svg)](https://mybinder.org/v2/gh/AdrianKriger/geo3D_wrkshp/HEAD?urlpath=%2Fdoc%2Ftree%2Fworkshop%2Fnotebooks%2CityJSONLoD1timing.ipynb)

<div class="alert alert-block alert-warning"><b>This notebook will:</b>

> **illustrate how different resolution elevation models (25m, 15m and 5m DEM) affect;**
>>
>> **1) `osm_LoD1_3DCityModel.ipynb` timing and**<br> 
>> **2) the quality** *(holes and completeness)* **of a LoD1 3D City Model**
</div>

In [1]:
import time
from datetime import timedelta
import tempfile

import os
from itertools import chain
import math

import requests
import overpass
import osm2geojson

import numpy as np
import pandas as pd
import geopandas as gpd
import topojson as tp

from shapely.geometry import polygon
from shapely.geometry import Point, Polygon 
from shapely.ops import snap
from shapely.ops import transform
from shapely.strtree import STRtree

import copy
import json

import pyproj

import city3D

from osgeo import gdal, ogr, osr

import triangle as tr

from openlocationcode import openlocationcode as olc

import matplotlib.pyplot as plt

In [2]:
Tstart = time.time()
import warnings
warnings.filterwarnings('ignore')

**A `parameter.json` defines the path and files**.

In [3]:
#jparams = json.load(open('osm3DsRiver_param10m.json'))  # 11 min 30 sec 
jparams = json.load(open('osm3DsRiver_param15m.json'))  # 5 min 20 sec
#jparams = json.load(open('osm3DsRiver_param25m.json'))  # 2 min 13 sec 

| area of interest | elevation model | CityJSON and metadata |
|:--------:|:--------:|:--------:|
|![area.png](_static/area.png)|![raster.png](_static/raster15.png)|![meta.png](_static/meta15.png) |

In [4]:
#- input OSM PBF file
input_pbf = "./data/CapeTown.osm.pbf"

**Lets first harvest the boundary of the area; we want to interogate**

In [5]:
#- get the area [suburb]
query = """[out:json][timeout:30];
        area[boundary=administrative][name='{0}'] -> .a;
        (
        way[amenity='university'][name='{1}'](area.a);
        relation[place][place~"sub|town|city|count|state|village|borough|quarter|neighbourhood"][name='{1}'](area.a);
        );
        out geom;
        """.format(jparams['LargeArea'], jparams['FocusArea'])

url = "http://overpass-api.de/api/interpreter"
r = requests.get(url, params={'data': query})
#rr = r.read()
area = osm2geojson.json2geojson(r.json())
#read into .gpd
aoi = gpd.GeoDataFrame.from_features(area['features'])
#aoi = aoi.set_crs(4326, allow_override=True)
if jparams['osm_type'] == 'relation' and len(aoi) > 1:
    for i, row in aoi.iterrows():
        if row.tags != None and 'place' in row.tags:
            focus = row
            
    trim = pd.DataFrame(focus)
    trim = trim.T
    aoi = gpd.GeoDataFrame(trim, geometry = trim['geometry'])
    #aoi = aoi.set_crs(4326)

# Drop rows where geometry is None or NaN
aoi = aoi.dropna(subset=['geometry'])
aoi = aoi.set_crs(4326, allow_override=True)

# gt the bounding box (BBOX) of the GeoJSON boundary
minx, miny, maxx, maxy = aoi.total_bounds
bbox_filter = "-spat", str(minx), str(miny), str(maxx), str(maxy)
aoi.head(2)

,geometry,type,id,tags
0,"MULTIPOLYGON (((18.47036 -33.93075, 18.47050 -...",relation,2034284,"{'name': 'Salt River', 'place': 'suburb', 'typ..."


**Only harvest what we need from the osm.pbf.**

In [6]:
start = time.time()

gdal.UseExceptions()
gdal.SetConfigOption("OGR_GEOMETRY_ACCEPT_UNCLOSED_RING", "NO") 
#gdal.SetConfigOption("USE_CUSTOM_INDEXING", "NO")
# Input OSM PBF file
#input_pbf = "your_data.osm.pbf"

# GDAL Virtual File System (VSI) to avoid writing to disk
geojson_vsimem = "/vsimem/temp.geojson"

#- GDAL VectorTranslate to extract only buildings & fix geometries
gdal.VectorTranslate(
    geojson_vsimem,                                           # Output as in-memory GeoJSON
    input_pbf,                                                # Source OSM PBF file
    format="GeoJSON",                                         # Output format
    layers=["multipolygons"],                                 # Extract only multipolygons
    options=["-where", "building IS NOT NULL", "-makevalid", 
                          "-spat", str(minx), str(miny), str(maxx), str(maxy)]  # Filter buildings & fix geometries
)

#- load into GeoDataFrame
gdf = gpd.read_file(geojson_vsimem)

#- cleanup VSI Memory
gdal.Unlink(geojson_vsimem)

# show gdf
#gdf.head()

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:00:01.543407


In [7]:
gdf.head(2)
#len(gdf)

,osm_id,osm_way_id,name,type,amenity,building,craft,historic,leisure,man_made,office,shop,sport,tourism,other_tags,geometry
0,6383946,NaN,Western Cape Metrorail - Infrastructure Building,multipolygon,NaN,office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""addr:city""=>""Cape Town"",""addr:postcode""=>""729...","MULTIPOLYGON (((18.47234 -33.92889, 18.47238 -..."
1,6691666,NaN,NaN,multipolygon,NaN,hall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""addr:housename""=>""Parish Hall""","MULTIPOLYGON (((18.46860 -33.93602, 18.46862 -..."


In [8]:
# Convert valid strings, ignore None/NaN
def safe_convert(tag_string):
    if isinstance(tag_string, str):
        try:
            # Replace "=>" with ":" and fix newlines
            formatted_string = "{" + tag_string.replace("=>", ":").replace("\n", " ") + "}"
            return json.loads(formatted_string)  # Parse safely
        except json.JSONDecodeError:
            return {}  # Return empty dict on failure
    return {}  # Return empty dict if NaN or None

# Apply conversion function
gdf["tags"] = gdf["other_tags"].apply(safe_convert)

# Extract values safely
#gdf["building"] = gdf["tags"].apply(lambda d: d.get("building", None) if isinstance(d, dict) else None)
gdf["building:levels"] = gdf["tags"].apply(lambda d: d.get("building:levels", None) if isinstance(d, dict) else None)
gdf["building:part"] = gdf["tags"].apply(lambda d: d.get("building:part", None) if isinstance(d, dict) else None)
#df["amenity"] = df["tags"].apply(lambda d: d.get("amenity", None) if isinstance(d, dict) else None)

gdf = gdf[gdf.geometry.apply(lambda x: x.within(aoi.unary_union))]
gdf.head(2)

,osm_id,osm_way_id,name,type,amenity,building,craft,historic,leisure,man_made,office,shop,sport,tourism,other_tags,geometry,tags,building:levels,building:part
0,6383946,NaN,Western Cape Metrorail - Infrastructure Building,multipolygon,NaN,office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""addr:city""=>""Cape Town"",""addr:postcode""=>""729...","MULTIPOLYGON (((18.47234 -33.92889, 18.47238 -...","{'addr:city': 'Cape Town', 'addr:postcode': '7...",1,None
7,13328172,NaN,NaN,multipolygon,NaN,school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""building:levels""=>""2"",""ref:ZA:emis""=>""1033101...","MULTIPOLYGON (((18.46158 -33.93174, 18.46176 -...","{'building:levels': '2', 'ref:ZA:emis': '10331...",2,None


In [9]:
ts = gdf[gdf['building'].notna()]
#len(ts)
print('\n', len(ts), "buildings have been harvested from", input_pbf)


 1451 buildings have been harvested from ./data/CapeTown.osm.pbf


In [10]:
ts.head(2)

,osm_id,osm_way_id,name,type,amenity,building,craft,historic,leisure,man_made,office,shop,sport,tourism,other_tags,geometry,tags,building:levels,building:part
0,6383946,NaN,Western Cape Metrorail - Infrastructure Building,multipolygon,NaN,office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""addr:city""=>""Cape Town"",""addr:postcode""=>""729...","MULTIPOLYGON (((18.47234 -33.92889, 18.47238 -...","{'addr:city': 'Cape Town', 'addr:postcode': '7...",1,None
7,13328172,NaN,NaN,multipolygon,NaN,school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""building:levels""=>""2"",""ref:ZA:emis""=>""1033101...","MULTIPOLYGON (((18.46158 -33.93174, 18.46176 -...","{'building:levels': '2', 'ref:ZA:emis': '10331...",2,None


In [11]:
#ts.head(2)

In [12]:
# basic cleaning to harvest building=* (no building:part=*) and building=levels tags only

ts.dropna(subset=['building:levels'], inplace= True)
ts['building:levels'] = pd.to_numeric(ts['building:levels'], downcast='integer')
ts['building:levels'] = ts['building:levels'].astype(int)
#- we only want buildings with =levels data
ts = ts[ts['building:levels'] != 0]
ts['building:levels'] = ts['building:levels'].replace('None', np.nan)#, inplace=True)
ts = ts[ts['building:levels'].notna()]

#- without building:part
ts = ts[ts['building:part'].isnull()]
#ts = ts.explode()
print('\n\033[1m', jparams['FocusArea'], 'has \033[0m', len(ts), 'buildings')


 Salt River has  1377 buildings


In [13]:
#- fill <Projected CRS: EPSG:32734> from above here epsg = EPSG:32734
epsg = 'EPSG:32734'

In [14]:
#project blds
ts = ts.to_crs(epsg)
#project aoi
aoi = aoi.to_crs(epsg)

**Create LoD1 3D City Model**

In [15]:
aoibuffer = aoi.copy()

def buffer01(row):
    with np.errstate(invalid='ignore'):
        return row.geometry.buffer(150, cap_style=3, join_style=2)

aoibuffer['geometry'] = aoibuffer.apply(buffer01, axis=1)
    
extent = [aoibuffer.total_bounds[0] - 250, aoibuffer.total_bounds[1] - 250, 
          aoibuffer.total_bounds[2] + 250, aoibuffer.total_bounds[3] + 250]

**Now the DEM**  
*one is available at [raster](https://github.com/AdrianKriger/geo3D/suburb/tree/main/raster)*

In [16]:
gdal.SetConfigOption("GTIFF_SRS_SOURCE", "GEOKEYS")
gdal.UseExceptions() 

# set the path and nodata
OutTile = gdal.Warp(jparams['projClip_raster'], 
                    jparams['in_raster'],
                    dstSRS=epsg,
                    srcNodata = jparams['nodata'],
                    #-  dstNodata = 0,
                    #-- outputBounds=[minX, minY, maxX, maxY]
                    outputBounds = [extent[0], extent[1], extent[2], extent[3]])
OutTile = None 

In [17]:
#- convert raster to XYZ in-memory
#- Virtual in-memory path
xyz_mem_path = "/vsimem/temp_xyz.xyz"  
gdal.Translate(xyz_mem_path, jparams['projClip_raster'], format="XYZ")  

#0 read XYZ from GDAL's in-memory file
xyz_vsimem = gdal.VSIFOpenL(xyz_mem_path, "rb")
xyz_bytes = gdal.VSIFReadL(1, gdal.VSIStatL(xyz_mem_path).size, xyz_vsimem)
gdal.VSIFCloseL(xyz_vsimem)
#- cleanup in-memory file
gdal.Unlink(xyz_mem_path) 

0

**prepare to harvest elevation**

In [18]:
# set the path to the projected, cliped elevation
src_filename = jparams['projClip_raster']

src_ds = gdal.Open(src_filename) 
gt_forward = src_ds.GetGeoTransform()
rb = src_ds.GetRasterBand(1)

def rasterQuery(geom, gt_forward, rb):

    mx = geom.representative_point().x
    my = geom.representative_point().y
    
    px = int((mx - gt_forward[0]) / gt_forward[1])
    py = int((my - gt_forward[3]) / gt_forward[5])

    intval = rb.ReadAsArray(px, py, 1, 1)
 
    return intval[0][0]

**Buildings**

In [19]:
#- harvest buildings
ts.drop(ts.index[ts['type'] == 'node'], inplace = True)

#- orient segments and simplify topology
topo = tp.Topology(ts, prequantize=False, winding_order='CCW_CW')
with np.errstate(invalid='ignore'):
    ts = topo.toposimplify(0.25).to_gdf()

In [20]:
# prepare to plot (more buildings = more time) 
start = time.time()

ts_copy = ts.copy()
new_df1 = ts_copy.loc[ts_copy.overlaps(ts_copy.unary_union)].reset_index(drop=True)  #-- perhaps no union?

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:00:11.059488


**Plot**

*Browse the saved `'./data/topologyFig'` at your leisure*

In [21]:
#%matplotlib

#fig, ax = plt.subplots(figsize=(11, 11))

#ts.plot(ax=ax, facecolor='none', edgecolor='purple', alpha=0.2)
#if len(new_df1) > 0:
#    new_df1.plot(ax=ax, edgecolor='red', facecolor='none')
#-- save
#plt.savefig('./data/topologyFig', dpi=300)
#plt.show()

In [22]:
#- get the mean height of the bld
ts['mean'] = ts.apply(lambda row: rasterQuery(row.geometry, gt_forward, rb), axis = 1)
ts.head(2)

,geometry,osm_id,osm_way_id,name,type,amenity,building,craft,historic,leisure,man_made,office,shop,sport,tourism,other_tags,tags,building:levels,building:part,mean
0,"MULTIPOLYGON (((266353.277 6242850.213, 266357...",6383946,NaN,Western Cape Metrorail - Infrastructure Building,multipolygon,NaN,office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""addr:city""=>""Cape Town"",""addr:postcode""=>""729...","{'addr:city': 'Cape Town', 'addr:postcode': '7...",1,None,4.315807
7,"MULTIPOLYGON (((265366.084 6242509.527, 265364...",13328172,NaN,NaN,multipolygon,NaN,school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""building:levels""=>""2"",""ref:ZA:emis""=>""1033101...","{'building:levels': '2', 'ref:ZA:emis': '10331...",2,None,16.917141


In [23]:
#ts['building:levels'].unique

In [24]:
# -- execute function. write geoJSON
city3D.write_geojson(ts, jparams)

In [25]:
start = time.time()

dis = gpd.read_file(jparams['osm_bldings'])                   
dis.set_crs(epsg=int(epsg[-5:]), inplace=True, allow_override=True)

dict_vertices = {}
cols = [c for c in ['bottom_bridge_height', 'bottom_roof_height', 'roof_height'] if c in dis.columns]

dis['geometry'] = dis.geometry.apply(polygon.orient, 1)

for i, row in dis.iterrows():
    oring = list(row.geometry.exterior.coords)
    name = row['osm_id']
    for (j, v) in enumerate(oring[:-1]):
        vertex = (oring[j][0], oring[j][1])
        attr = [row[c] for c in cols]
        attr = [x for x in attr if not np.isnan(x)]  # Remove np.nan values
        if vertex in dict_vertices.keys():
            dict_vertices[vertex][row['osm_id']] = attr
        else:
            dict_vertices[vertex] = {row['osm_id']: attr}

result = {}
for k1, d in dict_vertices.items():
    for k2 in d:
        result.setdefault(k2, {})[k1] = sorted(list(set([j for i in d.values() for j in i])))
        
dis.drop(dis.index[dis['building'] == 'bridge'], inplace = True)
dis.drop(dis.index[dis['building'] == 'roof'], inplace = True)

#- create a point representing the hole within each building  
dis['x'] = dis.representative_point().x
dis['y'] = dis.representative_point().y
hs = dis[['x', 'y', 'ground_height']].copy()

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:00:00.365914


In [26]:
print(len(dis), 'buildings have been harvested from the osm.pbf for the', jparams["FocusArea"], 'area') 

1372 buildings have been harvested from the osm.pbf for the Salt River area


In [27]:
dis.head(2)
#dis.plot()

,osm_id,address,building,building:use,building:levels,beds,building:flats,building:units,rooms,residential,social_facility,footprint,plus_code,ground_height,bottom_roof_height,building_height,roof_height,geometry,x,y
0,6383946,Western Cape Metrorail 7295 Cape Town,office,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'Polygon', 'coordinates': [[[266353.2...",CCXPX5XJ+X3V,4.32,NaN,4.1,8.42,"POLYGON ((266353.277 6242850.213, 266357.266 6...",266394.180182,6.242829e+06
1,13328172,NaN,school,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'Polygon', 'coordinates': [[[265366.0...",CHXWXPXM+X9W,16.92,NaN,6.9,23.82,"POLYGON ((265366.084 6242509.527, 265364.518 6...",265378.733465,6.242489e+06


## 1. Timing

In [28]:
#- 
dis_c = dis.copy()

In [29]:
#- prepare xyz (more buildings = more time)
start = time.time()

# Convert bytes to DataFrame
xyz_str = xyz_bytes.decode("utf-8")  # Decode to string

dtype_spec = {
    "x": np.float32,  # Reduce precision from float64 to float32 (saves memory)
    "y": np.float32,
    "z": np.float32
}

#df = pd.read_csv(jparams['xyz'], delimiter = ' ', header=None, names=["x", "y", "z"])
#df = pd.read_csv(jparams['xyz'], delimiter=" ", header=None, names=["x", "y", "z"], dtype=dtype_spec, engine="pyarrow")
                                                                                 #, reduced memory, parallelized reading)
df = pd.read_csv(pd.io.common.StringIO(xyz_str), delimiter=" ",  header=None, names=["x", "y", "z"], dtype=dtype_spec)


#geometry = [Point(xy) for xy in zip(df.x, df.y)]
geometry = gpd.points_from_xy(df.x, df.y)                                     #- vectorization faster
gdf = gpd.GeoDataFrame(df, crs=epsg, geometry=geometry)  

_symdiff = gpd.overlay(aoibuffer, dis_c, keep_geom_type=False, how='symmetric_difference')

#_mask = gdf.within(_symdiff.loc[0, 'geometry'])
#gdf = gdf.loc[_mask]

#- use STRtree for efficient spatial lookup instead of .within()
tree = STRtree(gdf.geometry)
possible_matches = tree.query(_symdiff.unary_union)         #- Faster than looping over individual geometries
#- filter candidate points with actual within() check
gdf = gdf.iloc[possible_matches]
gdf = gdf[gdf.within(_symdiff.unary_union)]

gdf = gdf[gdf['z'] != jparams['nodata']]                              
gdf.reset_index(drop=True, inplace=True)
gdf = gdf.round(2)

end = time.time()
print('runtime:', str(timedelta(seconds=(end - start))))

runtime: 0:04:56.923432


<div class="alert alert-block alert-info"><b>Why is this process taking so long?</b> 
</div>

*-- **What are we doing and why?***
|  | |
|:--------:|:--------:|
|**LoD1**: we only need the surface of the  <br> Earth where there are no buildings.<br><br> **Symmetric Difference**: we are removing all the points, harvested from the DEM, that fall inside a building. <br><br> These are essentially **spatial queries** <br> *—specifically spatial indexing and filtering*. <br><br>  Here we execute `STRtree` for efficient spatial operations. <br><br>  The greater the number of points *(higher resolution elevation model)* and the higher the number of buildings the longer the process will take! |![symDiff.png](_static/symDiff.png)|

In [30]:
dis.tail(2)

,osm_id,address,building,building:use,building:levels,beds,building:flats,building:units,rooms,residential,social_facility,footprint,plus_code,ground_height,bottom_roof_height,building_height,roof_height,geometry,x,y
1375,1313817370,NaN,yes,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'Polygon', 'coordinates': [[[265395.6...",CJXCXHX5+XCW,3.9,NaN,4.1,8.0,"POLYGON ((265395.618 6243201.900, 265382.518 6...",265388.558568,6.243198e+06
1376,1313817371,NaN,yes,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'Polygon', 'coordinates': [[[265297.7...",CFX3XHX5+XVV,4.4,NaN,4.1,8.5,"POLYGON ((265297.730 6243132.831, 265344.795 6...",265321.559683,6.243133e+06


<div class="alert alert-block alert-warning"><b>Prepare for Triangle:</b> </div>

The Python code to execute the `city3D.functions` are in the `city3D.py` script

In [31]:
#- harvest the building vertices. typically the corners. 
ac, c, min_zbld = city3D.getBldVertices(dis, gt_forward, rb)
idx = []
#- segments 
idx, idx01 = city3D.createSgmts(ac, c, gdf, idx)
#- populate the .df with coordinate values (the vertices)
df2 = city3D.concatCoords(gdf, ac)

#- do the same for the area of interest
acoi, ca = city3D.getAOIVertices(aoibuffer, gt_forward, rb)
idx, idx01 = city3D.createSgmts(acoi, ca, df2, idx)
df3 = city3D.concatCoords(df2, acoi)

**Triangle**

In [32]:
pv_pts = df3[['x', 'y', 'z']].values

In [33]:
holes01 = hs[['x', 'y']].round(3).values.tolist()
pts = df3[['x', 'y']].values #, 'z']].values

#the terrain without the blds
A = dict(vertices=pts, segments=idx, holes=holes01)

Tr = tr.triangulate(A, 'p')                  
terrTin = Tr.get('triangles').tolist()

**CityJSON**

In [34]:
#- 
minz = df3['z'].min()
maxz = df3['z'].max()

<div class="alert alert-block alert-warning"><b>create CityJSON</b> </div>

The Python code to execute the `.output_cityjson` function is in the `city3D.py` script

In [35]:
# -- execute function. create CityJSON
crs = epsg[5:]

city3D.output_cityjson(extent, minz, maxz, terrTin, pv_pts, jparams, min_zbld, acoi, result, crs) 

In [36]:
src_ds = None

<div class="alert alert-block alert-info"><b></b> 

**Go over to [Ninja the online CityJSON viewer](https://ninja.cityjson.org/#) and explore!**

</div>

In [37]:
Tend = time.time()
print('runtime:', str(timedelta(seconds=(Tend - Tstart))))

runtime: 0:05:17.446738


## 2. Quality

<div class="alert alert-block alert-danger"><b>WARNING:</b>  
    
***Lower resolution Elevation Models can leave gaps!***</div>

|  | |
|:--------:|:--------:|
|**25m**|![25.png](_static/25.png)|

<div class="alert alert-block alert-success"><b></b>

**Higher resolution Elevation Models do solve the challenge.** 
</div>

|  | |
|:--------:|:--------:|
|**15m**|![15.png](_static/15.png)|
|**10m**|![10.png](_static/10.png)|

<div class="alert alert-block alert-info"><b>Why is this happening?</b> 
</div>

|  | |
|:--------:|:--------:|
|**What do we** *(in the geospatial community)* **mean  when we say; <br><br> "3D"?**|![3dgis.png](_static/_3DGIS.png)|

|  | |
|:--------:|:--------:|
|We model terrain (a raster DEM) as a 2D surface imbedded in 3D space. <br><br> Each 'xy' coordinate (pixel) only has one 'z' height. This is typically called **2.5D modelling**. <br><br> Notice that *'truthfully'* representing objects connected to a ground surface is impossible. A wall for example could never be straight but have to deviate from the vertical. <br><br> This is why a DEM is often defined as the surface of the earth free of man-made and natural features |![25D.png](_static/_25D.png)|
|To represent a surface *'truthfully'* we can employ **2.75D modelling**. <br><br> The challenge with this solution is; it models the exterior only and it is one surface where objects are one feature. <br><br> A 3D mesh is a 2.75D surface and while traditionally a CAD tool its foray into GIS is recent|![275D.png](_static/_275D.png)|
|Full volumetric **3D modelling**, like a 3D City Model, is actually a 2.5D surface including volumetric 3D objects. <br><br>We can estimate BVPC from these models because we can calculate the volume of a structure|![3D.png](_static/_3D.png)|

<div class="alert alert-block alert-info"><b>Why is this important?</b> 
</div>

We model terrain seperately from the objects (trees, buildings, etc) connected to it.

We remove the buildings, roads, trees, etc. *---we cut them out as we did above--* and due to how *geo3D* creates a city model (terrain modelled seperate from the buildings); the resolution of the raster DEM can create challanges. 
<div class="alert alert-block alert-success"><b></b>

**The challenge is overcome with a finer resolution elevation model.** 
</div>
 

In this particular case a 15m DEM easily solves the challange.